# 03. Retrieval: ALS и ItemKNN

Сравнение MostPopular, ALS, ItemKNN, RRF и обычного объединения на одинаковых январских пользователях. Оценка проводится при одинаковых K; февральские ответы в этом эксперименте не используются.

## 1. Данные и окружение

Модели используют общий пакет movielens_recsys и настройки основного эксперимента.

In [1]:
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
from pathlib import Path
from time import perf_counter
import json
import pandas as pd
from IPython.display import display
from movielens_recsys.data import load_movielens
import numpy as np

# Обычный запуск: cwd = notebooks либо корень открытого workspace.
if (Path.cwd() / '../movielens-reccomender/configs/experiment.json').exists():
    ROOT = (Path.cwd() / '../movielens-reccomender').resolve()
elif (Path.cwd() / 'configs/experiment.json').exists():
    ROOT = Path.cwd()
else:
    ROOT = (Path.cwd() / 'ml-1m/movielens-reccomender').resolve()
config = json.loads((ROOT / 'configs/experiment.json').read_text(encoding='utf-8'))
ratings, movies, users = load_movielens(ROOT / config['data_dir'])
SEED = config['seed']
RETRIEVAL_K = config['evaluation']['retrieval_k']
FINAL_K = 10

## 2. История, ground truth и когорта

History — все оценки до 1 января 2001; validation — январь. Матрица предпочтений содержит только положительные оценки, seen — все оценки. Пользователи без положительной истории получают Popular fallback и остаются в общей оценке.

In [3]:
cutoff = pd.Timestamp(config['split']['validation_start'])
end = pd.Timestamp(config['split']['test_start'])
history = ratings.loc[ratings['datetime'] < cutoff].copy()
validation = ratings.loc[(ratings['datetime'] >= cutoff) & (ratings['datetime'] < end)].copy()
positive_history = history.loc[history['rating'] >= config['positive_threshold']]
seen_movies = history.groupby('user_id')['movie_id'].apply(set).to_dict()
ground_truth = (validation.loc[validation['rating'] >= config['positive_threshold']]
                .groupby('user_id')['movie_id'].apply(set).to_dict())
eligible_users = sorted(set(ground_truth) & set(history['user_id']))
catalog = set(history['movie_id'])
assert history['datetime'].max() < cutoff
assert all(ground_truth[uid].isdisjoint(seen_movies[uid]) for uid in eligible_users)
print(f'History: {len(history):,}; validation: {len(validation):,}; users: {len(eligible_users)}; catalog: {len(catalog)}')

History: 904,757; validation: 18,004; users: 506; catalog: 3678


## 3. Сравнение с MostPopular

Таблица загружается из ноутбука 02. В конце результаты MostPopular проверяются повторным вычислением на текущей когорте.

In [4]:
baseline_table = pd.read_csv(ROOT / "reports/baselines/mostpopular_validation.csv")
display(baseline_table)

,model,k,precision,recall,hitrate,ndcg,catalog_coverage,unique_recommended_items
0,MostPopular,10,0.091304,0.049683,0.359684,0.108800,0.036705,135
1,MostPopular,50,0.063083,0.171381,0.604743,0.128559,0.106308,391
2,MostPopular,100,0.051403,0.274495,0.717391,0.154932,0.167482,616
3,MostPopular,250,0.036071,0.438946,0.808300,0.202682,0.304513,1120


## 4. Разреженная user–item матрица

Маппинги user/movie ID построены по history. CSR-матрица имеет размер число исторических пользователей x число исторических фильмов. Единицы соответствуют положительным парам; нули означают отсутствие положительного наблюдения. Пользователи и фильмы без positive имеют нулевые строки или столбцы.

In [5]:
from scipy.sparse import csr_matrix
M = history['user_id'].nunique()
N = history['movie_id'].nunique()

user_mapper = dict(zip(np.unique(history["user_id"]), list(range(M))))
movie_mapper = dict(zip(np.unique(history["movie_id"]), list(range(N))))

user_inv_mapper = dict(zip(list(range(M)), np.unique(history["user_id"])))
movie_inv_mapper = dict(zip(list(range(N)), np.unique(history["movie_id"])))

user_index = [user_mapper[i] for i in history['user_id']]
item_index = [movie_mapper[i] for i in history['movie_id']]


positive_history = history[history["rating"] >= config['positive_threshold']]
positive_pairs = positive_history[['user_id','movie_id']].drop_duplicates()

values = np.ones(len(positive_pairs), dtype=np.float32,)
row_indices = (positive_pairs["user_id"].map(user_mapper).to_numpy())
column_indices = (positive_pairs["movie_id"].map(movie_mapper).to_numpy())

X = csr_matrix((values, (row_indices,column_indices)),shape=(len(user_mapper),
len(movie_mapper),),dtype=np.float32,) # за счет маперов по хистори, имеем 0
assert X.shape == (
    history["user_id"].nunique(),
    history["movie_id"].nunique(),
)
assert np.all(X.data == 1)

In [6]:
X

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 523365 stored elements and shape (6034, 3678)>

## 5. Implicit ALS

ALS обучается на положительных взаимодействиях. Маппинги переводят исходные ID во внутренние индексы и обратно. При выдаче исключаются все seen, включая оценки 1–3; для пользователей без положительной истории используется Popular fallback. Оценки разных моделей не складываются напрямую.

In [7]:
from implicit.als import AlternatingLeastSquares
als_model = AlternatingLeastSquares(**config["retrieval"]["als_parameters"],
use_gpu=False, random_state=SEED, calculate_training_loss=True)


als_fit_started = perf_counter()
als_model.fit(X)
als_fit_seconds = perf_counter() - als_fit_started


In [8]:
from movielens_recsys.retrieval import (
    recommend_als as _recommend_als,
    recommend_item_knn as _recommend_item_knn,
    recommend_popular as _recommend_popular,
)

def recommend_als(user_id, n_candidates=100):
    return _recommend_als(user_id, n_candidates, retrieval)

## 6. ItemKNN

Cosine similarity считается между фильмами по бинарным положительным взаимодействиям. Для каждого фильма остаются top-L положительных соседей, без self-similarity. Пользовательский score суммирует сходство с его положительными фильмами. L — число соседей фильма, M — число кандидатов пользователя; это разные параметры.

In [9]:
popularity = (
    positive_history
    .groupby("movie_id")
    .size()
    .reset_index(name="popularity")
    .sort_values(
        ["popularity", "movie_id"],
        ascending=[False, True],
    )
)

popular_movie_ids = popularity["movie_id"].tolist()

def recommend_popular(user_id, n_candidates):
    return _recommend_popular(user_id, n_candidates, retrieval)


In [10]:
knn_fit_started = perf_counter()
item_similarity = cosine_similarity(
    X.T.tocsr(),
    dense_output=True,
)
np.fill_diagonal(item_similarity, 0)
item_neighbors = {}
N_NEIGHBORS = config["retrieval"]["item_knn_neighbors"]
for item_idx in range(item_similarity.shape[0]):
    similarities = item_similarity[item_idx]
    sorted_indices = np.argsort(-similarities, kind="stable")
    neighbors = []
    for neighbor_idx in sorted_indices:
        similarity = similarities[neighbor_idx]
        if similarity <= 0:
            break

        neighbors.append(neighbor_idx)

        if len(neighbors) == N_NEIGHBORS:
            break
    item_neighbors[item_idx] = neighbors
print(item_similarity.shape)
knn_fit_seconds = perf_counter() - knn_fit_started

(3678, 3678)


In [ ]:
# Все объекты здесь обучены на одной и той же history.
# retrieval - обычный словарь, который передаёт данные функциям из src.
retrieval = {
    "X": X,
    "user_mapper": user_mapper,
    "movie_mapper": movie_mapper,
    "movie_inv_mapper": movie_inv_mapper,
    "seen_movies": seen_movies,
    "popular_movie_ids": popular_movie_ids,
    "popularity_scores": popularity.set_index("movie_id")["popularity"].to_dict(),
    "als_model": als_model,
    "item_similarity": item_similarity,
    "item_neighbors": item_neighbors,
}

def recommend_item_knn(user_id, n_candidates=100):
    return _recommend_item_knn(user_id, n_candidates, retrieval)


In [12]:
user_id = eligible_users[0]

knn_example = recommend_item_knn(
    user_id=user_id,
    n_candidates=10,
)

print(knn_example)
print(set(knn_example) & seen_movies[user_id])

[1198, 1196, 318, 1, 527, 593, 296, 1259, 1265, 1291]
set()


## 7. Общий evaluator

Все модели оцениваются на одной когорте, историческом каталоге и полном ground truth. NDCG использует все будущие positives в IDCG, включая пропущенные retrieval. Проверки исключают дубли, seen и неизвестные movie_id из выдачи.

In [13]:
popular_recommendations = {}
als_recommendations = {}
knn_recommendations = {}

for user_id in eligible_users:
    popular_recommendations[user_id] = recommend_popular(user_id, 250)
    als_recommendations[user_id] = recommend_als(user_id, 250)
    knn_recommendations[user_id] = recommend_item_knn(user_id, 250)

In [14]:
import math


def evaluate_recommendations(
    recommendations,
    ground_truth,
    eligible_users,
    catalog,
    model_name,
    k,
):
    assert k > 0 and len(eligible_users) > 0 and len(catalog) > 0
    rows = []

    for user_id in eligible_users:
        recs = recommendations[user_id][:k]
        relevant = ground_truth[user_id]
        assert len(relevant) > 0
        assert len(recs) == len(set(recs)), (model_name, user_id, "duplicates")
        assert set(recs).issubset(catalog), (model_name, user_id, "catalog")

        hits = sum(
            movie_id in relevant
            for movie_id in recs
        )

        precision = hits / k
        recall = hits / len(relevant)
        hitrate = float(hits > 0)

        dcg = 0

        for rank, movie_id in enumerate(recs, start=1):
            if movie_id in relevant:
                dcg += 1 / math.log2(rank + 1)

        ideal_hits = min(len(relevant), k)

        idcg = sum(
            1 / math.log2(rank + 1)
            for rank in range(1, ideal_hits + 1)
        )

        ndcg = dcg / idcg if idcg > 0 else 0

        rows.append({
            "user_id": user_id,
            "recommendation_count": len(recs),
            "precision": precision,
            "recall": recall,
            "hitrate": hitrate,
            "ndcg": ndcg,
        })

    metrics_by_user = pd.DataFrame(rows)

    unique_recommended = {
        movie_id
        for user_id in eligible_users
        for movie_id in recommendations[user_id][:k]
    }

    result = {
        "model": model_name,
        "k": k,
        "precision": metrics_by_user["precision"].mean(),
        "recall": metrics_by_user["recall"].mean(),
        "hitrate": metrics_by_user["hitrate"].mean(),
        "ndcg": metrics_by_user["ndcg"].mean(),
        "catalog_coverage": len(unique_recommended) / len(catalog),
        "mean_list_length": metrics_by_user["recommendation_count"].mean(),
        "users": len(eligible_users),
    }

    return metrics_by_user, result

In [15]:
union_recommendations = {}

for user_id in eligible_users:
    candidates = []
    candidate_scores = {}
    lists = [
        als_recommendations[user_id][:100],
        knn_recommendations[user_id][:100],
        popular_recommendations[user_id][:50],
    ]

    for recommendations in lists:
        for rank, movie_id in enumerate(recommendations, start=1):
            if movie_id not in candidate_scores:
                candidate_scores[movie_id] = 0

            candidate_scores[movie_id] += 1 / (60 + rank)
    sorted_candidates = sorted(
        candidate_scores.items(),
        key=lambda pair: (-pair[1], pair[0]),
    )
    for movie_id, score in sorted_candidates:
        candidates.append(movie_id)
    for movie_id in popular_recommendations[user_id]:
        if len(candidates) >= 250:
            break

        if movie_id not in candidates:
            candidates.append(movie_id)

    union_recommendations[user_id] = candidates

### Обычное объединение: два отдельных эксперимента
1. `Union_ALS_first+Popular`: тот же состав ALS100 + KNN100 + Popular50 с той же добивкой до 250, но порядок ALS → KNN → Popular. Recall@250 обязан совпасть с RRF: меняется только порядок.
2. `Union_ALS150_KNN100+Popular`: первые 150 ALS и первые 100 KNN, затем добивка Popular. Это уже другой бюджет источников. Первые 150 остаются в порядке ALS; сравнение @10/50/100 с ALS будет одинаковым по построению.

Склейка списков — явно заданный порядок, а не обученное ранжирование. Полное объединение ALS250 и KNN250 может содержать до 500 фильмов: его нельзя выдавать за сравнение на одинаковом бюджете 250.

In [16]:
union_als_first_recommendations = {}
union_als_knn_recommendations = {}
raw_union_lengths = []

for user_id in eligible_users:
    candidates = []
    lists = [
        als_recommendations[user_id][:100],
        knn_recommendations[user_id][:100],
        popular_recommendations[user_id][:50],
    ]
    for recommendations in lists:
        for movie_id in recommendations:
            if movie_id not in candidates:
                candidates.append(movie_id)
    raw_union_lengths.append(len(candidates))

    for movie_id in popular_recommendations[user_id]:
        if len(candidates) >= config["retrieval"]["max_candidates"]:
            break
        if movie_id not in candidates:
            candidates.append(movie_id)
    union_als_first_recommendations[user_id] = candidates

    # Новый состав: больше мест ALS, без начальной квоты Popular.
    candidates = []
    for movie_id in als_recommendations[user_id][:config["retrieval"]["als_candidates"]]:
        if movie_id not in candidates:
            candidates.append(movie_id)
    for movie_id in knn_recommendations[user_id][:config["retrieval"]["item_knn_candidates"]]:
        if movie_id not in candidates:
            candidates.append(movie_id)
    for movie_id in popular_recommendations[user_id]:
        if len(candidates) >= config["retrieval"]["max_candidates"]:
            break
        if movie_id not in candidates:
            candidates.append(movie_id)
    union_als_knn_recommendations[user_id] = candidates

    # При том же составе RRF не может потерять relevant в полном пуле.
    assert set(union_recommendations[user_id]) == set(
        union_als_first_recommendations[user_id]
    )

print("Сырой union до добивки: средняя длина", np.mean(raw_union_lengths))
print("Состав RRF и ALS-first совпадает для всех пользователей.")

Сырой union до добивки: средняя длина 152.6324110671937
Состав RRF и ALS-first совпадает для всех пользователей.


### Проверки перед сравнением
History и маппинги берутся до cutoff. Ground truth не ограничиваем известным каталогом: будущие cold items остаются промахами. Все модели оцениваются на одной когорте. Короткие списки не скрываем.
Для ALS/KNN при равных scores используем стабильный порядок индексов; маппинг фильмов построен по отсортированным movie_id.

In [17]:
assert X.nnz == len(positive_pairs)
assert X.shape == (len(user_mapper), len(movie_mapper))
assert len(item_neighbors) == X.shape[1]
assert all(movie_inv_mapper[idx] == mid for mid, idx in movie_mapper.items())
for item_idx, neighbors in item_neighbors.items():
    assert item_idx not in neighbors
    assert len(neighbors) <= N_NEIGHBORS
    assert all(item_similarity[item_idx, idx] > 0 for idx in neighbors)

for recommender in [recommend_popular, recommend_als, recommend_item_knn]:
    assert recommender(eligible_users[0], 0) == []
    assert recommender(-1, 10) == recommend_popular(-1, 10)
no_positive_users = np.flatnonzero(X.getnnz(axis=1) == 0)
for user_idx in no_positive_users:
    uid = user_inv_mapper[user_idx]
    assert recommend_als(uid, 10) == recommend_popular(uid, 10)
    assert recommend_item_knn(uid, 10) == recommend_popular(uid, 10)

checked_models = {
    "MostPopular": popular_recommendations,
    "ALS": als_recommendations,
    "ItemKNN": knn_recommendations,
    "Union_RRF+Popular": union_recommendations,
    "Union_ALS_first+Popular": union_als_first_recommendations,
    "Union_ALS150_KNN100+Popular": union_als_knn_recommendations,
}
for name, recommendations in checked_models.items():
    assert set(recommendations) == set(eligible_users)
    for uid in eligible_users:
        recs = recommendations[uid]
        assert len(recs) == len(set(recs)), (name, uid, "duplicates")
        assert set(recs).isdisjoint(seen_movies[uid]), (name, uid, "seen")
        assert set(recs).issubset(catalog), (name, uid, "catalog")
        assert len(recs) == 250, (name, uid, len(recs))
print("Проверены все модели: когорта, каталог, seen, дубли, длина 250.")

Проверены все модели: когорта, каталог, seen, дубли, длина 250.


In [18]:
models = {
    "MostPopular": popular_recommendations,
    "ALS": als_recommendations,
    "ItemKNN": knn_recommendations,
    "Union_RRF+Popular": union_recommendations,
    "Union_ALS_first+Popular": union_als_first_recommendations,
    "Union_ALS150_KNN100+Popular": union_als_knn_recommendations
}

results = []

for k in [10, 50, 100, 250]:
    for model_name, recommendations in models.items():
        _, result = evaluate_recommendations(
            recommendations=recommendations,
            ground_truth=ground_truth,
            eligible_users=eligible_users,
            catalog=catalog,
            model_name=model_name,
            k=k,
        )

        results.append(result)

comparison = pd.DataFrame(results)

display(comparison)

,model,k,precision,recall,hitrate,ndcg,catalog_coverage,mean_list_length,users
0,MostPopular,10,0.091304,0.049683,0.359684,0.108800,0.036705,10.0,506
1,ALS,10,0.092688,0.075491,0.446640,0.116128,0.146003,10.0,506
2,ItemKNN,10,0.107708,0.061298,0.387352,0.128906,0.060087,10.0,506
3,Union_RRF+Popular,10,0.102372,0.063582,0.420949,0.121714,0.059543,10.0,506
4,Union_ALS_first+Popular,10,0.092688,0.075491,0.446640,0.116128,0.146003,10.0,506
5,Union_ALS150_KNN100+Popular,10,0.092688,0.075491,0.446640,0.116128,0.146003,10.0,506
6,MostPopular,50,0.063083,0.171381,0.604743,0.128559,0.106308,50.0,506
7,ALS,50,0.064901,0.218735,0.671937,0.147732,0.269984,50.0,506
8,ItemKNN,50,0.069289,0.164426,0.569170,0.139000,0.172376,50.0,506
9,Union_RRF+Popular,50,0.072332,0.204968,0.640316,0.147483,0.181892,50.0,506


## 9. Объединение источников

Исторический RRF объединяет ALS100 + KNN100 + Popular50 с добивкой до 250. Основной вариант — ALS150 + KNN100, затем Popular-добивка. При полном пуле важен состав; при небольшом K — также порядок. Вклад KNN определяется дополнительными релевантными кандидатами сверх ALS.

In [19]:
lengths = []

for user_id in eligible_users:
    lengths.append(len(union_recommendations[user_id]))

print("Средняя длина:", np.mean(lengths))
print("Минимальная:", min(lengths))
print("Максимальная:", max(lengths))

Средняя длина: 250.0
Минимальная: 250
Максимальная: 250


In [20]:
by_user, union_result = evaluate_recommendations(
    recommendations=union_recommendations,
    ground_truth=ground_truth,
    eligible_users=eligible_users,
    catalog=catalog,
    model_name="Union",
    k=250,
)

print(union_result)

{'model': 'Union', 'k': 250, 'precision': np.float64(0.03842687747035574), 'recall': np.float64(0.49808875303919575), 'hitrate': np.float64(0.8438735177865613), 'ndcg': np.float64(0.22653642230150275), 'catalog_coverage': 0.4032082653616096, 'mean_list_length': np.float64(250.0), 'users': 506}


In [21]:
user_id = eligible_users[0]

als_candidates = set(als_recommendations[user_id][:100])
knn_candidates = set(knn_recommendations[user_id][:100])

new_knn_candidates = knn_candidates - als_candidates
new_relevant = new_knn_candidates & ground_truth[user_id]

print("Новые положительные фильмы от KNN:", new_relevant)

Новые положительные фильмы от KNN: set()


## Итоговая проверка и вклад KNN

Добавленные и потерянные релевантные фильмы сравниваются с ALS@250. Таблица сравнения сохраняется для ноутбука 04; остальные промежуточные результаты остаются в выводах.

In [22]:
for saved in baseline_table.to_dict(orient="records"):
    row = comparison[(comparison["model"] == "MostPopular") & (comparison["k"] == saved["k"])].iloc[0]
    for metric in ["precision", "recall", "hitrate", "ndcg", "catalog_coverage"]:
        assert np.isclose(row[metric], saved[metric]), (saved["k"], metric)

contributions = []
for name, recommendations in models.items():
    added = 0
    lost = 0
    macro_recall_delta = []
    for uid in eligible_users:
        original = set(als_recommendations[uid][:250])
        alternative = set(recommendations[uid][:250])
        relevant = ground_truth[uid]
        new_hits = len((alternative - original) & relevant)
        lost_hits = len((original - alternative) & relevant)
        added += new_hits
        lost += lost_hits
        macro_recall_delta.append((new_hits - lost_hits) / len(relevant))
    contributions.append({
        "model": name, "added_relevant_vs_ALS250": added,
        "lost_relevant_vs_ALS250": lost,
        "recall_delta_vs_ALS250": np.mean(macro_recall_delta),
    })
display(pd.DataFrame(contributions))
print("Baseline воспроизведён. ALS fit seconds:", round(als_fit_seconds, 3))
print("ItemKNN similarity + neighbors seconds:", round(knn_fit_seconds, 3))

report_dir = ROOT / "reports/retrieval"
report_dir.mkdir(parents=True, exist_ok=True)
comparison.to_csv(report_dir / "validation.csv", index=False)

,model,added_relevant_vs_ALS250,lost_relevant_vs_ALS250,recall_delta_vs_ALS250
0,MostPopular,1653,1589,-0.074498
1,ALS,0,0,0.000000
2,ItemKNN,1403,1064,-0.052950
3,Union_RRF+Popular,1415,1053,-0.015356
4,Union_ALS_first+Popular,1415,1053,-0.015356
5,Union_ALS150_KNN100+Popular,1189,770,0.005957


Baseline воспроизведён. ALS fit seconds: 0.813
ItemKNN similarity + neighbors seconds: 0.924
